# 03 · Model karşılaştırma (10 model: 2 sıfırdan CNN, 7 transfer, 1 ViT)

**Önce GPU açın.** Colab: `Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU`. Kaggle: sağ panel → *Session options* → *Accelerator* → GPU ve *Internet: On*. Bu notebook her iki platformda da çalışır.

Akış:
1. Kurulum, GPU kontrolü, veri
2. Hızlı deneme (≈5 dk): indirme ve eğitim döngüsü sorunsuz mu?
3. **Dengeleme karşılaştırması** (yalnızca validation): ağırlıksız CE, ağırlıklı CE, focal, ağırlıklı örnekleme → kazanan tüm modellerde kullanılır
4. 10 modelin eğitimi (seed 42). **Biten koşular atlanır**: oturum kopsa bile aynı notebook'u yeniden çalıştırmanız yeterli
5. Özet tablo

Test kümesi bu notebook'ta **hiç yüklenmez**.

In [ ]:
import os, sys, glob, shutil, zipfile
ON_KAGGLE = os.path.exists('/kaggle/working')

if ON_KAGGLE:
    # ---- Kaggle: kod ve önceki çıktılar /kaggle/input altındaki veri setlerinden gelir (salt okunur) ----
    hits = glob.glob('/kaggle/input/**/src/config.py', recursive=True)
    assert hits, 'Kod veri seti bulunamadı: sağ panelden Add Input ile yzproje-code veri setini ekleyin.'
    code_root = os.path.dirname(os.path.dirname(hits[0]))
    PROJECT = '/kaggle/working/YZProje'
    shutil.copytree(code_root, PROJECT, dirs_exist_ok=True)          # kod her oturumda tazelenir
    os.makedirs(PROJECT + '/outputs', exist_ok=True)
    out_hits = glob.glob('/kaggle/input/**/splits/all_files.csv', recursive=True)
    assert out_hits, 'Çıktı veri seti bulunamadı: Add Input ile yzproje-outputs veri setini ekleyin.'
    out_src = os.path.dirname(os.path.dirname(out_hits[0]))
    # yalnızca çalışma klasöründe OLMAYAN dosyaları kopyala (oturum içi yeni sonuçlar ezilmesin)
    shutil.copytree(out_src, PROJECT + '/outputs', dirs_exist_ok=True,
                    copy_function=lambda s, d: None if os.path.exists(d) else shutil.copy2(s, d))
    os.makedirs('/kaggle/temp/data', exist_ok=True)
    os.environ['KNEE_DATA_ROOT'] = '/kaggle/temp/data'                # ham veri: geçici disk (çıktıya sayılmaz)
else:
    # ---- Colab: Drive ----
    from google.colab import drive
    drive.mount('/content/drive')
    # Projeyi Drive'da bul (iç içe klasör olsa bile); zip açılmamışsa aç
    hits = glob.glob('/content/drive/MyDrive/**/src/config.py', recursive=True)
    if not hits:
        zips = glob.glob('/content/drive/MyDrive/**/YZProje*.zip', recursive=True)
        if zips:
            zipfile.ZipFile(zips[0]).extractall('/content/drive/MyDrive/')
            hits = glob.glob('/content/drive/MyDrive/**/src/config.py', recursive=True)
    if not hits:
        raise FileNotFoundError('Drive içinde src/config.py bulunamadı.')
    roots = [os.path.dirname(os.path.dirname(h)) for h in hits]
    # Birden fazla kopya varsa, veri listeleri (outputs/splits) olan klasör tercih edilir
    with_out = [r for r in roots if os.path.exists(f'{r}/outputs/splits/all_files.csv')]
    PROJECT = sorted(with_out or roots, key=len)[0]
    os.environ['KNEE_DATA_ROOT'] = '/content/data'

os.chdir(PROJECT); sys.path.insert(0, PROJECT)
os.environ['KNEE_OUTPUT_DIR'] = f'{PROJECT}/outputs'
print('Platform:', 'Kaggle' if ON_KAGGLE else 'Colab', '| Proje klasörü:', PROJECT)
print(sorted(os.listdir(PROJECT + '/outputs')) if os.path.exists(PROJECT + '/outputs') else 'outputs yok (ilk çalıştırma)')

In [ ]:
!pip install -q imagehash==4.3.2 tabulate==0.9.0 timm==1.0.30
import torch
assert torch.cuda.is_available(), 'GPU yok! Colab: Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU | Kaggle: sağ panel Session options > Accelerator > GPU'
print(torch.__version__, torch.cuda.get_device_name(0))
!mkdir -p outputs/env && pip freeze > outputs/env/pip_freeze_train.txt

In [ ]:
# Veri oturum diskinde yoksa yeniden indir (Colab oturumu kapanınca /content/data silinir)
DATA_URL = ''   # boş bırakabilirsiniz: 01 notebook'unda kullandığınız bağlantı Drive'daki kayıttan okunur
ZIP_PATH = ''
from pathlib import Path
from src import config as C, data_prep
if not (DATA_URL or ZIP_PATH):
    _info = Path(C.AUDIT_DIR) / 'download_info.json'
    if _info.exists():
        import json
        DATA_URL = json.load(open(_info)).get('download_url', '')
        if DATA_URL: print('Veri bağlantısı kayıttan okundu.')
if any(Path(C.EXTRACT_DIR).rglob('kneeKL224')) if Path(C.EXTRACT_DIR).exists() else False:
    print('Veri hazır:', data_prep.get_dataset_root())
else:
    if not (DATA_URL or ZIP_PATH):
        raise ValueError('Veri bu oturumda yok: DATA_URL veya ZIP_PATH doldurun.')
    # Sabit dosya listeleri Drive'da duruyor; yalnızca görüntüler yeniden açılır.
    data_prep.main(['--url', DATA_URL] if DATA_URL else ['--zip', ZIP_PATH])
    print('Veri hazır:', data_prep.get_dataset_root())

## Doğrulanan yakın-kopya çiftleri (görsel)

4 çiftin neye benzediğini görmek için (yaklaşık 1 dk).

In [ ]:
from src import dup_verify
from IPython.display import Image, display
dup_verify.main([])
display(Image(filename=str(C.FIG_DIR / 'audit_confirmed_duplicates.png'), width=900))

## Hızlı deneme

Gerçek ImageNet ağırlıklarının inmesini ve eğitim döngüsünü küçük alt kümede (1 epoch) dener.

In [ ]:
from src.train import train_model
for k in ['EfficientNetB0', 'SimpleCNN']:
    train_model(k, 42, name=f'{k}__quick', subset_fraction=0.1, max_epochs_override=1, force=True)

## Dengeleme karşılaştırması (yalnızca validation)

EfficientNet-B0 ile 4 seçenek (her biri ≈10–15 dk). Kazanan `outputs/balancing_choice.json` dosyasına yazılır.

In [ ]:
from src.train import balancing_comparison, get_chosen_loss
import json, pandas as pd
if not (C.OUTPUT_DIR / 'balancing_choice.json').exists():
    balancing_comparison('EfficientNetB0')
ch = json.load(open(C.OUTPUT_DIR / 'balancing_choice.json'))
display(pd.DataFrame(ch['results']).T.round(4)); print('Seçilen kayıp:', ch['chosen_loss'])

## 10 modelin eğitimi

Sıra hızlıdan yavaşa. Her koşu bitince Drive'a yazılır.

In [ ]:
from src.models import ZOO
ORDER = ['EfficientNetB0', 'MobileNetV3L', 'SimpleCNN', 'MiniResNet', 'ResNet50', 'DenseNet121',
         'ConvNeXtTiny', 'DeiTSmall', 'Xception', 'InceptionV3']
assert set(ORDER) == set(ZOO) - {'Original'}
LOSS = get_chosen_loss(); print('Ortak kayıp:', LOSS)
for k in ORDER:
    train_model(k, 42, loss=LOSS)

## Özet (validation)

In [ ]:
r = pd.read_csv(C.OUTPUT_DIR / 'results.csv')
r = r[~r.model_name.str.contains('__')]
cols = ['model_name','param_count','epochs','train_time_s','latency_ms_median','val_accuracy',
        'val_balanced_accuracy','val_macro_f1','val_qwk','val_mae_grade','val_roc_auc_macro']
display(r[cols].sort_values('val_macro_f1', ascending=False).round(4))

`outputs/results.csv` ve `outputs/balancing_choice.json` dosyalarını, ayrıca sonuç tablosunu paylaşın.